In [ ]:
###
# セットアップ
###

from pathlib import Path
import json
import sys

import torch
from IPython.display import display
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as F

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_PATH = Path("/content/drive/MyDrive/cnn-hands-on")
except Exception:
    ROOT_PATH = Path.cwd()

if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("ROOT_PATH:", ROOT_PATH)
print("device:", device)

In [ ]:
###
# Cell 01
# Roboflowのデータセット取得コードを貼り付ける
###



print("dataset:", dataset.location)

In [ ]:
###
# Cell 02
# COCO形式のファイルを確認する
###

DATASET_DIR = Path(dataset.location)

for split in ["train", "valid", "test"]:
    annotation_path = DATASET_DIR / split / "_annotations.coco.json"
    image_dir = DATASET_DIR / split
    print(split)
    print("  images:", image_dir)
    print("  annotation:", annotation_path)
    print("  exists:", annotation_path.exists())

In [ ]:
###
# Cell 03
# COCO形式の検出データセットをPyTorch Datasetとして読み込む
###

class COCODetectionDataset(Dataset):
    def __init__(self, root_dir, split="train"):
        self.root_dir = Path(root_dir)
        self.split = split
        self.split_dir = self.root_dir / split
        self.annotation_path = self.split_dir / "_annotations.coco.json"

        if not self.annotation_path.exists():
            raise FileNotFoundError(f"annotation file not found: {self.annotation_path}")

        coco = json.loads(self.annotation_path.read_text())
        self.images = sorted(coco["images"], key=lambda x: x["id"])
        self.image_id_to_annotations = {image["id"]: [] for image in self.images}

        for ann in coco["annotations"]:
            self.image_id_to_annotations.setdefault(ann["image_id"], []).append(ann)

        categories = sorted(coco["categories"], key=lambda x: x["id"])
        self.category_id_to_label = {
            category["id"]: index + 1
            for index, category in enumerate(categories)
        }
        self.label_to_name = {
            index + 1: category["name"]
            for index, category in enumerate(categories)
        }

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image_info = self.images[index]
        image_path = self.split_dir / image_info["file_name"]
        image = Image.open(image_path).convert("RGB")
        image_tensor = F.to_tensor(image)

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in self.image_id_to_annotations.get(image_info["id"], []):
            x, y, width, height = ann["bbox"]
            if width <= 0 or height <= 0:
                continue

            boxes.append([x, y, x + width, y + height])
            labels.append(self.category_id_to_label[ann["category_id"]])
            areas.append(width * height)
            iscrowd.append(ann.get("iscrowd", 0))

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([image_info["id"]], dtype=torch.int64),
            "area": torch.tensor(areas, dtype=torch.float32),
            "iscrowd": torch.tensor(iscrowd, dtype=torch.int64),
        }

        if target["boxes"].numel() == 0:
            target["boxes"] = torch.zeros((0, 4), dtype=torch.float32)

        return image_tensor, target

train_dataset = COCODetectionDataset(DATASET_DIR, split="train")
val_split = "valid" if (DATASET_DIR / "valid" / "_annotations.coco.json").exists() else "train"
val_dataset = COCODetectionDataset(DATASET_DIR, split=val_split)

print("train images:", len(train_dataset))
print("val images:", len(val_dataset))
print("classes:", train_dataset.label_to_name)

In [ ]:
###
# Cell 04
# 検出用DataLoaderを作成する
###

# 1枚ごとに物体数が違うため、通常のDataLoaderのまとめ方は使わない。
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn,
)

images, targets = next(iter(train_loader))
print("images in batch:", len(images))
print("image shape:", images[0].shape)
print("target keys:", targets[0].keys())
print("boxes shape:", targets[0]["boxes"].shape)
print("labels:", targets[0]["labels"])

In [ ]:
###
# Cell 05
# torchvisionの事前学習済み検出モデルを読み込む
###

from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_Weights,
    fasterrcnn_resnet50_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)

# background + Roboflowで作成したクラス数
num_classes = 1 + len(train_dataset.label_to_name)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model = model.to(device)

print("num_classes:", num_classes)
print(model.roi_heads.box_predictor)

In [ ]:
###
# Cell 06
# 転移学習を行う
###

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for images, targets in train_loader:
        images = [image.to(device) for image in images]
        targets = [
            {key: value.to(device) for key, value in target.items()}
            for target in targets
        ]

        loss_dict = model(images, targets)
        loss = sum(value for value in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"epoch {epoch + 1}/{num_epochs} loss: {avg_loss:.4f}")

In [ ]:
###
# Cell 07
# 検証画像で推論結果を可視化する
###

def draw_prediction(image_tensor, prediction, label_to_name, score_threshold=0.5):
    image = F.to_pil_image(image_tensor).convert("RGB")
    draw = ImageDraw.Draw(image)

    boxes = prediction["boxes"].detach().cpu()
    labels = prediction["labels"].detach().cpu()
    scores = prediction["scores"].detach().cpu()

    for box, label, score in zip(boxes, labels, scores):
        if score < score_threshold:
            continue
        x1, y1, x2, y2 = box.tolist()
        name = label_to_name.get(int(label), str(int(label)))
        text = f"{name} {score:.2f}"
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, max(0, y1 - 12)), text, fill="red")

    return image

model.eval()
sample_images, sample_targets = next(iter(val_loader))

with torch.no_grad():
    predictions = model([image.to(device) for image in sample_images])

for image, prediction in zip(sample_images[:4], predictions[:4]):
    display(draw_prediction(image, prediction, train_dataset.label_to_name, score_threshold=0.5))

In [ ]:
###
# Cell 08
# IoUを計算する関数
###

def box_iou_one_to_many(box, boxes):
    if boxes.numel() == 0:
        return torch.zeros((0,), dtype=torch.float32)

    x1 = torch.maximum(box[0], boxes[:, 0])
    y1 = torch.maximum(box[1], boxes[:, 1])
    x2 = torch.minimum(box[2], boxes[:, 2])
    y2 = torch.minimum(box[3], boxes[:, 3])

    intersection = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    box_area = (box[2] - box[0]).clamp(min=0) * (box[3] - box[1]).clamp(min=0)
    boxes_area = (boxes[:, 2] - boxes[:, 0]).clamp(min=0) * (boxes[:, 3] - boxes[:, 1]).clamp(min=0)
    union = box_area + boxes_area - intersection

    return intersection / union.clamp(min=1e-6)

In [ ]:
###
# Cell 09
# Precision / Recall / APを計算する
###

def evaluate_detection(model, data_loader, iou_threshold=0.5, score_threshold=0.05):
    model.eval()
    predictions_by_class = {}
    targets_by_class = {}

    with torch.no_grad():
        for images, targets in data_loader:
            outputs = model([image.to(device) for image in images])

            for target, output in zip(targets, outputs):
                image_id = int(target["image_id"].item())

                for label in target["labels"].tolist():
                    targets_by_class.setdefault(label, {}).setdefault(image_id, [])

                for box, label in zip(target["boxes"], target["labels"]):
                    targets_by_class.setdefault(int(label), {}).setdefault(image_id, []).append(box.cpu())

                boxes = output["boxes"].detach().cpu()
                labels = output["labels"].detach().cpu()
                scores = output["scores"].detach().cpu()

                for box, label, score in zip(boxes, labels, scores):
                    if score < score_threshold:
                        continue
                    predictions_by_class.setdefault(int(label), []).append(
                        {
                            "image_id": image_id,
                            "box": box,
                            "score": float(score),
                        }
                    )

    metrics = {}

    for label, targets_per_image in targets_by_class.items():
        predictions = sorted(
            predictions_by_class.get(label, []),
            key=lambda item: item["score"],
            reverse=True,
        )
        total_targets = sum(len(boxes) for boxes in targets_per_image.values())
        matched = {image_id: set() for image_id in targets_per_image}

        tp = []
        fp = []

        for prediction in predictions:
            image_id = prediction["image_id"]
            gt_boxes = targets_per_image.get(image_id, [])

            if not gt_boxes:
                tp.append(0)
                fp.append(1)
                continue

            gt_tensor = torch.stack(gt_boxes)
            ious = box_iou_one_to_many(prediction["box"], gt_tensor)
            best_iou, best_index = ious.max(dim=0)
            best_index = int(best_index)

            if best_iou >= iou_threshold and best_index not in matched[image_id]:
                tp.append(1)
                fp.append(0)
                matched[image_id].add(best_index)
            else:
                tp.append(0)
                fp.append(1)

        if predictions:
            tp_cum = torch.tensor(tp, dtype=torch.float32).cumsum(dim=0)
            fp_cum = torch.tensor(fp, dtype=torch.float32).cumsum(dim=0)
            precision_curve = tp_cum / (tp_cum + fp_cum).clamp(min=1e-6)
            recall_curve = tp_cum / max(total_targets, 1)

            # Precision-Recall曲線の面積を台形近似で計算する簡易AP
            recall_points = torch.cat([torch.tensor([0.0]), recall_curve])
            precision_points = torch.cat([torch.tensor([precision_curve[0]]), precision_curve])
            ap = torch.trapz(precision_points, recall_points).item()
            precision = precision_curve[-1].item()
            recall = recall_curve[-1].item()
        else:
            precision = 0.0
            recall = 0.0
            ap = 0.0

        metrics[label] = {
            "class": train_dataset.label_to_name.get(label, str(label)),
            "targets": total_targets,
            "predictions": len(predictions),
            "precision": precision,
            "recall": recall,
            "ap": ap,
        }

    mAP = sum(item["ap"] for item in metrics.values()) / max(len(metrics), 1)
    return metrics, mAP

metrics, mAP = evaluate_detection(model, val_loader, iou_threshold=0.5)

for label, item in metrics.items():
    print(
        item["class"],
        f"targets={item['targets']}",
        f"predictions={item['predictions']}",
        f"precision={item['precision']:.3f}",
        f"recall={item['recall']:.3f}",
        f"AP@0.5={item['ap']:.3f}",
    )

print(f"mAP@0.5: {mAP:.3f}")

In [ ]:
###
# Cell 10
# 学習した重みを保存する
###

SAVE_PATH = ROOT_PATH / "models" / "face_fasterrcnn_resnet50_fpn.pth"
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "label_to_name": train_dataset.label_to_name,
        "num_classes": num_classes,
    },
    SAVE_PATH,
)

print("saved:", SAVE_PATH)